In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("/content/MP_consolidated_regression_classification_dataset.csv")

print("Original size:", df.shape)


Original size: (154870, 19)


In [ ]:
df = df.drop_duplicates(subset=[
    "formula_pretty",
    "crystal_system",
    "spacegroup_number",
    "point_group"
])

print("After duplicate removal:", df.shape)


After duplicate removal: (134719, 19)


In [ ]:
df = df[
    (df["band_gap"] >= 0) & (df["band_gap"] <= 12) &
    (df["formation_energy_per_atom"] >= -10) & (df["formation_energy_per_atom"] <= 5) &
    (df["energy_above_hull"] >= 0) & (df["energy_above_hull"] <= 2)
]

print("After physical filtering:", df.shape)


After physical filtering: (132891, 19)


In [ ]:
df = df[df["nelements"] <= 5]
print("After nelements filtering:", df.shape)


After nelements filtering: (128004, 19)


In [ ]:
df["bg_bin"] = pd.cut(df["band_gap"], bins=10)

df_balanced = (
    df.groupby("bg_bin", group_keys=False)
      .apply(lambda x: x.sample(
          n=min(len(x), 3000),
          random_state=42
      ))
)

df_balanced = df_balanced.drop(columns=["bg_bin"])

print("After distribution-aware sampling:", df_balanced.shape)


After distribution-aware sampling: (17475, 19)


/tmp/ipython-input-159792469.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("bg_bin", group_keys=False)
/tmp/ipython-input-159792469.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(


In [ ]:
def bandgap_class(bg):
    if bg == 0:
        return "Metal"
    elif bg <= 3:
        return "Semiconductor"
    else:
        return "Insulator"

df_balanced["bg_class"] = df_balanced["band_gap"].apply(bandgap_class)


In [ ]:
REG_FEATURES = [
    "formula_pretty",
    "avg_atomic_number",
    "avg_electronegativity",
    "valence_electron_count",
    "fraction_transition_metals",
    "nelements",
    "density",
    "volume",
    "nsites",
    "crystal_system",
    "spacegroup_number",
    "point_group",
    "formation_energy_per_atom",
    "energy_above_hull",
    "band_gap"
]

regression_gold = df_balanced[REG_FEATURES]
regression_gold.to_csv("regression_gold.csv", index=False)


In [ ]:
CLS_FEATURES = [
    "formula_pretty",
    "avg_atomic_number",
    "avg_electronegativity",
    "valence_electron_count",
    "fraction_transition_metals",
    "nelements",
    "density",
    "volume",
    "nsites",
    "crystal_system",
    "spacegroup_number",
    "point_group",
    "bg_class"
]

classification_gold = df_balanced[CLS_FEATURES]
classification_gold.to_csv("classification_gold.csv", index=False)


In [ ]:
df_balanced.sample(n=2000, random_state=42).to_csv(
    "fast_debug.csv", index=False
)
